# Rekonstrukce makroskopických polí D2Q9

Distribuční funkce $f_q(x,y)$ obsahují devět populací D2Q9 v každé buňce. Hustota a rychlost se získají z nultého a prvního momentu:

$$\rho = \sum_{q=0}^{8} f_q,$$

$$u_x = \frac{1}{\rho}\sum_{q=0}^{8} f_q c_{qx}, \qquad u_y = \frac{1}{\rho}\sum_{q=0}^{8} f_q c_{qy}.$$

Tlak izotermálního D2Q9 modelu je:

$$p = c_s^2\rho = \frac{\rho}{3}.$$

Pořadí směrů odpovídá implementaci v `LBM/include/D2Q9.hpp`.

In [2]:
import numpy as np

#       0  E  N   W   S  NE  NW  SW  SE
CX = np.array([0, 1, 0, -1,  0, 1, -1, -1,  1], dtype=float)
CY = np.array([0, 0, 1,  0, -1, 1,  1, -1, -1], dtype=float)
WEIGHTS = np.array([4/9, 1/9, 1/9, 1/9, 1/9,
                    1/36, 1/36, 1/36, 1/36], dtype=float)
CS2 = 1.0 / 3.0


## Funkce `reconstruct_fields(f)`

Funkce přijímá pole ve tvaru `(9, ny, nx)` nebo `(ny, nx, 9)`. Vrací `(density, velocity, pressure)`. Hustota a tlak mají tvar `(ny, nx)`, zatímco vektorové rychlostní pole má tvar `(ny, nx, 2)`. Pro buňky s nulovou hustotou nastaví rychlost na nulu, aby nevzniklo dělení nulou.

In [3]:
def reconstruct_fields(f):
    """Reconstruct density and velocity fields from D2Q9 populations.

    Parameters
    ----------
    f : array_like
        D2Q9 populations with shape ``(9, ny, nx)`` or
        ``(ny, nx, 9)``.

    Returns
    -------
    density : numpy.ndarray
        Density field with shape ``(ny, nx)``.
    velocity : numpy.ndarray
        Vector velocity field with shape ``(ny, nx, 2)``. The final
        axis contains ``(ux, uy)``.
    pressure : numpy.ndarray
        Isothermal lattice pressure ``CS2 * density`` with shape
        ``(ny, nx)``.
    """
    populations = np.asarray(f, dtype=float)

    if populations.ndim != 3:
        raise ValueError(
            "f must be a 3D array with shape (9, ny, nx) or (ny, nx, 9)"
        )

    if populations.shape[0] == 9:
        fq = populations
    elif populations.shape[-1] == 9:
        fq = np.moveaxis(populations, -1, 0)
    else:
        raise ValueError(
            "one axis of f must contain exactly 9 D2Q9 populations"
        )

    if not np.all(np.isfinite(fq)):
        raise ValueError("f contains NaN or infinite values")

    density = np.sum(fq, axis=0)
    momentum_x = np.einsum("q,qyx->yx", CX, fq)
    momentum_y = np.einsum("q,qyx->yx", CY, fq)

    ux = np.zeros_like(density)
    uy = np.zeros_like(density)
    np.divide(momentum_x, density, out=ux, where=density != 0.0)
    np.divide(momentum_y, density, out=uy, where=density != 0.0)

    velocity = np.stack((ux, uy), axis=-1)
    pressure = CS2 * density

    return density, velocity, pressure


## Ověření

Rovnovážná klidová distribuce $f_q=w_q\rho$ musí rekonstruovat $\rho=1$ a nulovou rychlost v celé mřížce.

In [7]:
ny, nx = 4, 6
f_equilibrium = WEIGHTS[:, None, None] * np.ones((9, ny, nx))

density, velocity, pressure = reconstruct_fields(f_equilibrium)

assert np.allclose(density, 1.0)
assert np.allclose(velocity, 0.0)
assert np.allclose(pressure, 1.0 / 3.0)

print("density shape:", density.shape)
print("velocity shape:", velocity.shape)
print("pressure shape:", pressure.shape)
print("density min/max:", density.min(), density.max())
print("max |u|:", np.linalg.norm(velocity, axis=-1).max())
print("pressure min/max:", pressure.min(), pressure.max())


density shape: (4, 6)
velocity shape: (4, 6, 2)
pressure shape: (4, 6)
density min/max: 1.0000000000000002 1.0000000000000002
max |u|: 0.0
pressure min/max: 0.33333333333333337 0.33333333333333337


## Použití

```python
density, velocity, pressure = reconstruct_fields(f)
speed = np.linalg.norm(velocity, axis=-1)
```

`velocity[..., 0]` obsahuje $u_x$, `velocity[..., 1]` obsahuje $u_y$ a `speed` je velikost rychlosti $\sqrt{u_x^2+u_y^2}$.